# cellmap-flow on Colab (inference only) → HF Static frontend

The dashboard + Neuroglancer come from a pre-hosted static page; Colab
just runs the inference server. One cloudflared tunnel. No nginx,
no NG-Python, no dashboard Flask in Colab.

```
HF Static Space                        Colab
   • dashboard.html                       • cellmap_flow_server (Flask, port 8765)
   • neuroglancer.js (bundled)            • cloudflared (exposes 8765)
   • runs in YOUR browser

Your browser fetches:
   1. dashboard HTML + JS    → from HF (fast, S3-backed CDN)
   2. raw zarr chunks        → direct from public S3 (fast)
   3. inference chunks       → from Colab via cloudflared (T4 GPU)
```

The dashboard at `https://ackermand-cellmap-flow-demo.static.hf.space/dashboard.html` accepts URL params:
- `backend=<url>` — your Colab cloudflared inference URL
- `dataset=<slug>` — the model name (used as path on the inference server)
- `raw=<https-zarr-url>` — adds a raw EM layer fetched direct from S3

This notebook prints a one-click URL with those params pre-filled.

**Limit**: the cloudflared URL is public for the lifetime of this
Colab session. Anyone with the URL can hit your GPU. That's fine for
showing a colleague; don't post it on Twitter.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**.
2. **Run all cells.** If the install cell restarts the kernel, click Run All again.
3. Click the printed URL.

## 1. Install

In [ ]:
# Skip pip if already installed (after first-run kernel restart,
# this turns the install cell into a ~5s no-op).
def _all_packages_installed():
    try:
        import cellmap_flow.globals
        import bioimageio.core, bioimageio.spec
        return (
            bioimageio.core.__version__ == "0.9.6"
            and bioimageio.spec.__version__.startswith("0.5.7")
        )
    except Exception:
        return False

import os, subprocess

if _all_packages_installed():
    print("[install] python packages already at required versions — skipping pip.")
else:
    print("[install] running pip (slow first-run step) ...")
    %pip install -q "cellmap-flow[bioimageio] @ git+https://github.com/janelia-cellmap/cellmap-flow.git@browser-inference" huggingface_hub s3fs
    %pip install -q --force-reinstall "bioimageio.core==0.9.6" "bioimageio.spec==0.5.7.4"

# cloudflared — public tunnel from Colab → trycloudflare URL so the
# HF-hosted dashboard's NG-JS can fetch inference chunks from us.
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("[install] downloading cloudflared ...")
    subprocess.check_call([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared",
    ])
    subprocess.check_call(["chmod", "+x", "/usr/local/bin/cloudflared"])
    print("[install] cloudflared installed.")
else:
    print("[install] cloudflared already present.")

# Pre-download the default BMZ model so the first inference request
# doesn't pay the model-download cost on top of model load.
try:
    print("[install] pre-fetching bioimageio model 'hiding-blowfish' (cached on re-run) ...")
    import bioimageio.spec
    bioimageio.spec.load_description("hiding-blowfish")
    print("[install] model pre-fetched.")
except Exception as e:
    print(f"[install] could not pre-fetch model (will lazy-load on first inference): {e}")

# Verify the kernel can import cellmap_flow without numpy-ABI drama.
# Pip can mutate numpy on disk while Colab pre-imported a different
# version at startup. If those mismatch the next import will crash;
# force a kernel restart so Run-All on the second pass picks up
# correctly. (Won't trigger on the second pass.)
try:
    import cellmap_flow.globals
    print("[install] kernel is in sync, ready.")
except Exception as e:
    print(f"[install] kernel state mismatched on-disk packages ({e.__class__.__name__}: {e}).")
    print("[install] restarting kernel in 3s — when it comes back, click Run All again.")
    import time
    time.sleep(3)
    os.kill(os.getpid(), 9)

## 2. Configure model + dataset

`MODEL_TYPE = "huggingface"` for a cellmap HF model, or `"bioimage"`
for a BMZ model.

T4 sizing notes:
- 2D BMZ models (`hiding-blowfish`): trivial, fast.
- 178³ HF models (`fly_organelles_run07_*`): fit cleanly but slow on T4 (~1-2s/chunk).
- 288³ HF models (`jrc_mus-livers_*`): borderline, OOM-prone.

In [ ]:
MODEL_TYPE = "bioimage"      # or "huggingface"

# --- Mode A: huggingface ---
HF_REPO = "cellmap/fly_organelles_run07_432000"
HF_NAME = HF_REPO.split("/")[-1]
HF_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_mus-liver/jrc_mus-liver.zarr/recon-1/em/fibsem-uint8"
)

# --- Mode B: bioimage (BMZ) ---
BMZ_MODEL = "hiding-blowfish"
BMZ_VOXEL_SIZE = "8,8,8"
BMZ_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_hela-2/jrc_hela-2.zarr/recon-1/em/fibsem-uint8/s1"
)

INFERENCE_PORT = 8765

if MODEL_TYPE == "huggingface":
    MODEL_NAME = HF_NAME
    DATASET = HF_DATASET
elif MODEL_TYPE == "bioimage":
    MODEL_NAME = BMZ_MODEL
    DATASET = BMZ_DATASET
else:
    raise ValueError(f"unknown MODEL_TYPE={MODEL_TYPE!r}")

# Raw layer URL (the dashboard's ?raw= param). We strip /sN from the
# inference dataset path so NG points at the multiscale GROUP and
# picks levels itself, rather than locking us to one scale.
import re
RAW_URL = re.sub(r"/s\d+/?$", "", DATASET)

print(f"MODEL_TYPE = {MODEL_TYPE}")
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"DATASET    = {DATASET}")
print(f"RAW_URL    = {RAW_URL}")

## 3. Start the inference server + warm up the GPU

In [ ]:
import os, subprocess, time, requests

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if MODEL_TYPE == "huggingface":
    cmd = [
        "cellmap_flow_server", "huggingface",
        "--repo", HF_REPO, "--name", HF_NAME,
        "-d", HF_DATASET, "--port", str(INFERENCE_PORT),
    ]
elif MODEL_TYPE == "bioimage":
    cmd = [
        "cellmap_flow_server", "bioimage",
        "--model-name", BMZ_MODEL, "--voxel-size", BMZ_VOXEL_SIZE,
        "--name", BMZ_MODEL,
        "-d", BMZ_DATASET, "--port", str(INFERENCE_PORT),
    ]
print("starting:", " ".join(cmd))
server = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ},
)
print(f"server pid={server.pid}, waiting for it to listen on :{INFERENCE_PORT} ...")
for _ in range(300):
    line = server.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    if "Running on" in line or f":{INFERENCE_PORT}" in line:
        print("\n[server] ready.")
        break

# Warm up: hit a chunk URL once on localhost so the GPU loads the
# model + runs its first forward pass while later cells set up the
# tunnel. By the time the user clicks the dashboard URL, GPU is hot.
print("[warm-up] triggering a chunk to load model + warm GPU ...")
t0 = time.time()
try:
    r = requests.get(
        f"http://localhost:{INFERENCE_PORT}/{MODEL_NAME}/s0/0.0.0.0",
        timeout=180,
    )
    print(f"[warm-up] done in {time.time()-t0:.1f}s (status {r.status_code}, {len(r.content)} bytes)")
except Exception as e:
    print(f"[warm-up] failed (not fatal — will warm on first real request): {e}")

## 4. Open a cloudflared tunnel for the inference server

In [ ]:
import subprocess, re, time

def start_cloudflared(port: int, label: str, timeout: float = 60) -> tuple[str, subprocess.Popen]:
    """Start `cloudflared tunnel --url http://localhost:<port>` and parse
    the public trycloudflare URL from its stdout. Returns (url, process).
    Caller keeps the process alive — the tunnel dies when it exits."""
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}",
         "--no-autoupdate", "--metrics", "127.0.0.1:0"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    pat = re.compile(r"https://[-a-z0-9]+\.trycloudflare\.com")
    deadline = time.time() + timeout
    while time.time() < deadline:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                raise RuntimeError(f"cloudflared exited early (rc={proc.returncode})")
            time.sleep(0.1); continue
        m = pat.search(line)
        if m:
            print(f"[{label}] {m.group(0)}")
            return m.group(0), proc
    proc.terminate()
    raise RuntimeError(f"cloudflared did not emit a URL within {timeout}s")

INFERENCE_URL, _infer_tunnel = start_cloudflared(INFERENCE_PORT, "inference")
print(f"\nINFERENCE_URL = {INFERENCE_URL}")

## 5. Open the demo

In [ ]:
import urllib.parse

params = urllib.parse.urlencode({
    "backend":  INFERENCE_URL,
    "dataset":  MODEL_NAME,    # used as the layer's zarr path on the inference server
    "raw":      RAW_URL,       # raw EM streamed direct from S3 alongside inference
})
DEMO_URL = f"https://ackermand-cellmap-flow-demo.static.hf.space/dashboard.html?{params}"

print()
print("=" * 70)
print("Open this URL — that's the whole demo:")
print()
print(f"  {DEMO_URL}")
print()
print("Dashboard chrome + Neuroglancer load from HF (static).")
print(f"Raw layer streams direct from S3 (fast, no Colab in the path):")
print(f"  {RAW_URL}")
print("Inference layer goes through this Colab session's cloudflared tunnel:")
print(f"  {INFERENCE_URL}")
print("=" * 70)

## 6. Keep-alive

Leave this cell running. It drains server logs as they come in and
keeps the kernel alive so the inference server + cloudflared stay up.
Stop with ▢ to tear everything down.

In [ ]:
import time, select

def drain(proc, label):
    while True:
        r, _, _ = select.select([proc.stdout], [], [], 0)
        if not r: return
        line = proc.stdout.readline()
        if not line: return
        print(f"[{label}] {line}", end="")

try:
    while True:
        drain(server, "server")
        if server.poll() is not None:
            drain(server, "server")
            print(f"\n[server] exited rc={server.returncode}.")
            break
        time.sleep(2)
finally:
    try: server.terminate()
    except Exception: pass